In [ ]:
%pip install -q numpy==2.3.5 pandas==2.2.3 pyarrow==26.0.0 holidays==0.106 catboost==1.2.10 lightgbm==4.7.0 xgboost==3.1.3 scipy==1.16.3 scikit-learn==1.6.1


In [ ]:
from pathlib import Path
import subprocess,sys
Path('/tmp/course_worker.py').write_text('"""Private course-only fixed-count worker; existing approved inputs only."""\nimport base64,gc,hashlib,io,json,sys,zipfile,warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nPAYLOAD=\'...\'\nTAG=\'ll\'\nPHASE=\'confirmation\'\nSELECTED=\'wx_knn6\'\n\ndef main():\n    root=Path(\'/tmp/x5_course_\'+TAG);parent=root/\'Forecasting\';here=parent/\'course_only\';here.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as z:\n        for name in z.namelist():\n            target=parent/name if name in [\'notebook_api.py\',\'run_validation.py\'] else here/name\n            assert target.resolve().is_relative_to(root)\n            target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name))\n    source=list(Path(\'/kaggle/input\').rglob(\'input_manifest.json\'));assert len(source)==1;source=source[0].parent\n    raw_dir=root/\'Data/Shops\';raw_dir.mkdir(parents=True,exist_ok=True);baseline=parent/\'baseline\';baseline.mkdir(exist_ok=True)\n    (raw_dir/\'train_2.csv\').symlink_to(source/\'train_2.csv\');(baseline/\'original_solution.ipynb\').symlink_to(source/\'original_solution.ipynb\')\n    prepared=list(Path(\'/kaggle/input\').rglob(\'prepared_stage2.zip\'));assert len(prepared)==1\n    cache=parent/\'cache\';cache.mkdir(exist_ok=True)\n    with zipfile.ZipFile(prepared[0]) as z:\n        assert z.testzip() is None\n        for name in z.namelist():\n            if name.startswith((\'2024-08/\',\'2024-11/\',\'2024-12/\')):\n                target=(cache/name).resolve();assert target.is_relative_to(cache);target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name))\n    sys.path.insert(0,str(parent));sys.path.insert(0,str(here))\n    from notebook_api import MODELS,load_recipe,read_raw,compact_fold\n    from run_validation import get_fold\n    from runner import prepare,execute,march\n    model=MODELS[{\'cat\':0,\'ld\':1,\'ll\':2,\'xgb\':3}[TAG]]\n    expected=json.loads((here/\'preparation_checks.json\').read_text());assert expected[\'status\']==\'passed\'\n    assert hashlib.sha256((here/\'runner.py\').read_bytes()).hexdigest()==expected[\'implementation_sha256\']\n    assert hashlib.sha256((here/\'cache/weights_manifest.json\').read_bytes()).hexdigest()==expected[\'weights_manifest_sha256\']\n    output=Path(\'/kaggle/working\');archive=output/(PHASE+\'_\'+TAG+\'.zip\')\n    def export():\n        with zipfile.ZipFile(archive,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n            for folder,base,prefix in [(here/\'results\',here,\'\'),(parent/\'submissions/2025-03-course\',parent/\'submissions/2025-03-course\',\'march/\')]:\n                for path in folder.rglob(\'*\'):\n                    if path.is_file():z.write(path,prefix+path.relative_to(base).as_posix())\n    parity=None\n    try:\n        if PHASE==\'development\':\n            recipe=load_recipe();compact=get_fold(recipe,2024,11)\n            for config in [\'wx_knn6\',\'wx_knn12\']:\n                xt,xv,_,_,_,columns,hashes=prepare(recipe,compact,model,config)\n                assert columns==expected[\'november_expected\'][model][config][\'columns\']\n                assert hashes==expected[\'november_expected\'][model][config][\'WX_fingerprints\'],(model,config,hashes)\n                del xt,xv;gc.collect()\n            del compact;gc.collect()\n            execute(model,\'2024-11\',\'baseline\')\n            actual=pd.read_csv(here/\'results/2024-11\'/model/\'baseline.csv\').sort_values(\'new_id\')\n            with np.load(here/\'expected_baseline.npz\') as anchor:\n                np.testing.assert_array_equal(actual.new_id,anchor[\'ids\'])\n                np.testing.assert_array_equal(actual.pto,anchor[\'truth\'])\n                np.testing.assert_allclose(actual.prediction,anchor[model],rtol=1e-12,atol=1e-8)\n                parity=dict(max_relative_difference=float(np.max(abs(actual.prediction/anchor[model]-1))),\n                    mape=float((abs(actual.prediction-actual.pto)/actual.pto).mean()*100))\n            print(\'COURSE_BASELINE_PARITY_PASSED\',model,parity,flush=True);export()\n            for fold in [\'2024-08\',\'2024-11\',\'2024-12\']:\n                for config in [\'wx_knn6\',\'wx_knn12\']:execute(model,fold,config);export()\n        else:\n            assert PHASE==\'confirmation\' and SELECTED in [\'wx_knn6\',\'wx_knn12\']\n            for fold in [\'2025-01\',\'2025-02\']:\n                for config in [\'baseline\',SELECTED]:execute(model,fold,config);export()\n            recipe=load_recipe();raw=read_raw(recipe)\n            with warnings.catch_warnings():\n                warnings.simplefilter(\'ignore\',pd.errors.PerformanceWarning)\n                original=recipe.build_retail_features_with_split(raw,split_mode=\'train_val_test\',min_year=2023,\n                    val_year=2025,val_month=2,test_year=2025,test_month=3,create_test_if_missing=True,\n                    expected_test_size=18657,use_val_as_test_history=True,base_feature_kwargs=recipe.COMMON_BASE_FEATURE_KWARGS,verbose=False)\n            compact=compact_fold(recipe,original);del original,raw;gc.collect()\n            full=cache/\'final_2025-03\';full.mkdir(exist_ok=True)\n            expected_frames=json.loads((here/\'expected_march_frames.json\').read_text())\n            for key in [\'test_train_features\',\'test_features\']:\n                digest=hashlib.sha256(pd.util.hash_pandas_object(compact[key],index=False).values.tobytes()).hexdigest()\n                assert digest==expected_frames[key],key\n                compact[key].to_parquet(full/(key+\'.parquet\'),index=False)\n            (full/\'manifest.json\').write_text(json.dumps({\'categorical_cols\':compact[\'categorical_cols\'],\'feature_cols\':compact[\'feature_cols\']}))\n            del compact;gc.collect();march(model,SELECTED);export()\n        result=dict(status=\'passed\',tag=TAG,model=model,phase=PHASE,selected=SELECTED if PHASE==\'confirmation\' else None,\n            baseline_parity=parity,original_training_counts_preserved=True,private_existing_datasets_only=True,\n            runner_sha256=hashlib.sha256((here/\'runner.py\').read_bytes()).hexdigest(),\n            weights_manifest_sha256=expected[\'weights_manifest_sha256\'])\n        (output/(PHASE+\'_\'+TAG+\'_checks.json\')).write_text(json.dumps(result,indent=2))\n    finally:export()\n    print(\'COURSE_PHASE_DONE\',PHASE,TAG,flush=True)\n\nif __name__==\'__main__\':main()\n')
subprocess.run([sys.executable,'-u','/tmp/course_worker.py'],check=True)
